# knee-mri-detect — train on Colab

Runtime → Change runtime type → **GPU (T4 or better)**.

You need a Kaggle API token: kaggle.com → Settings → API → *Create New Token* (downloads `kaggle.json`). Upload it when cell 2 asks.

In [ ]:
!git clone https://github.com/Akhil-Prasad09/knee-mri-detect.git
%cd knee-mri-detect
!pip install -q -r requirements.txt
!nvidia-smi -L

In [ ]:
from google.colab import files
import os, shutil
os.makedirs('/root/.kaggle', exist_ok=True)
up = files.upload()  # choose kaggle.json
shutil.move(next(iter(up)), '/root/.kaggle/kaggle.json'); os.chmod('/root/.kaggle/kaggle.json', 0o600)
!kaggle datasets download cjinny/mrnet-v1 -p data/raw --unzip
!ls data/raw/MRNet-v1.0 && ls data/raw/MRNet-v1.0/train/sagittal | wc -l

Trains sagittal → coronal → axial with `ml/training/config.yaml` (EfficientNet-B3, 20 epochs). ~2–3 h on a T4. Best validation-AUC checkpoint per plane is kept.

In [ ]:
!python -m ml.training.train --config ml/training/config.yaml

In [ ]:
!python -m ml.training.evaluate --config ml/training/config.yaml
!cat ml/models/eval.json

Download `models.zip`, unzip into `ml/models/` in the repo (`*.pt` are git-ignored; `eval.json` is too — keep both locally), then `make api`.

In [ ]:
!zip -j models.zip ml/models/*.pt ml/models/*.json
files.download('models.zip')